# insurance-cv

**Temporal cross-validation for insurance pricing models -- walk-forward splits that respect policy year, accident year, and IBNR development structure.**

Standard k-fold cross-validation overestimates model quality on insurance data by 10-15% because future development data leaks into training. This notebook demonstrates the problem, shows how walk-forward CV fixes it, and benchmarks the difference on synthetic UK motor data with a known trend.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-cv/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-cv polars scikit-learn numpy

## 1. Synthetic UK motor portfolio

Three years of policies with a +15% per year claims trend. The trend is the critical feature: k-fold averages across all time periods and misses it; walk-forward reveals it fold by fold.

In [ ]:
import polars as pl
import numpy as np
from datetime import date, timedelta

rng = np.random.default_rng(42)
n = 3_000
start = date(2021, 1, 1)

# Spread policies evenly across 3 years
inception_dates = [
    start + timedelta(days=int(d))
    for d in rng.integers(0, 365 * 3, n)
]

# Quarter index for each policy (0 = 2021 Q1)
quarter_idx = np.array([
    (d.year - 2021) * 4 + (d.month - 1) // 3
    for d in inception_dates
])

# True frequency: base 0.08, +15% pa = +3.75% per quarter (compounding)
true_freq = 0.08 * (1.0375 ** quarter_idx)
claim_count = rng.poisson(true_freq).astype(int)

df = pl.DataFrame({
    "policy_id": [f"POL{i:05d}" for i in range(n)],
    "inception_date": inception_dates,
    "exposure": rng.uniform(0.3, 1.0, n).round(3).tolist(),
    "claim_count": claim_count.tolist(),
    "vehicle_age": rng.integers(0, 12, n).tolist(),
    "driver_age": rng.integers(18, 75, n).tolist(),
    "ncd_years": rng.integers(0, 9, n).tolist(),
}).with_columns(pl.col("inception_date").cast(pl.Date))

print(f"Portfolio: {df.shape[0]:,} policies")
print(f"Date range: {df['inception_date'].min()} to {df['inception_date'].max()}")
print(f"Claim frequency: {df['claim_count'].sum() / n:.4f} overall")
print(f"True trend: +15% pa (known DGP)")
df.head(5)

## 2. Generate walk-forward splits

`walk_forward_split` creates an expanding training window with a rolling test window. The `ibnr_buffer_months` parameter excludes claims in the months immediately before the test window -- these are partially developed and contaminate your test evaluation.

In [ ]:
from insurance_cv import walk_forward_split, temporal_leakage_check, split_summary

splits = walk_forward_split(
    df,
    date_col="inception_date",
    min_train_months=12,   # need at least a year to cover seasonality
    test_months=6,         # evaluate on 6-month windows
    step_months=6,         # non-overlapping test periods
    ibnr_buffer_months=3,  # exclude 3 months before test window (motor IBNR)
)

# Always validate before fitting -- catches forward-looking splits
check = temporal_leakage_check(splits, df, date_col="inception_date")
if check["errors"]:
    raise RuntimeError("Temporal leakage detected: " + "\n".join(check["errors"]))
print("Leakage check: PASS (no forward-looking splits)\n")

summary = split_summary(splits, df, date_col="inception_date")
print(summary)

## 3. Walk-forward CV vs random k-fold: the benchmark

We fit a Poisson regression on each fold and compare CV scores. Walk-forward gives a higher (worse-looking) Poisson deviance -- but it is the honest prospective estimate. K-fold's lower score is an artefact of temporal leakage.

In [ ]:
from insurance_cv import InsuranceCV
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import cross_val_score, KFold
import warnings

features = ["vehicle_age", "driver_age", "ncd_years"]
X = df.select(features).to_numpy()
y = df["claim_count"].to_numpy().astype(float)

model = PoissonRegressor(max_iter=300)

# Walk-forward CV (insurance-cv)
cv_temporal = InsuranceCV(splits, df)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    scores_temporal = cross_val_score(
        model, X, y,
        cv=cv_temporal,
        scoring="neg_mean_poisson_deviance",
    )

# Random k-fold (the wrong way)
kfold = KFold(n_splits=len(splits), shuffle=True, random_state=42)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    scores_kfold = cross_val_score(
        model, X, y,
        cv=kfold,
        scoring="neg_mean_poisson_deviance",
    )

mean_temporal = -scores_temporal.mean()
mean_kfold = -scores_kfold.mean()

print("=== CV Score Comparison (Poisson deviance, lower = better-looking) ===")
print(f"Walk-forward (insurance-cv): {mean_temporal:.5f}")
print(f"Random k-fold:               {mean_kfold:.5f}")
print(f"K-fold optimism:             {(mean_temporal - mean_kfold) / mean_temporal:.1%} lower than walk-forward")
print()
print("K-fold looks better because future data leaks into training.")
print("The walk-forward estimate is what you will actually see in production.")

## 4. Fold-by-fold trajectory

Walk-forward gives you a timeline, not just a number. The per-fold deviance should rise as the test window advances deeper into the trending period. K-fold folds are shuffled -- no temporal pattern, no warning signal.

In [ ]:
print("Walk-forward per-fold trajectory:")
print(f"{'Fold':>4}  {'Deviance':>10}  {'Trend signal'}")
for i, s in enumerate(-scores_temporal):
    arrow = " (rising: model degrades as trend advances)" if i > 0 and s > -scores_temporal[i-1] else ""
    print(f"  {i+1:2d}   {s:9.5f}{arrow}")

print()
print("Random k-fold per-fold scores (no temporal ordering):")
for i, s in enumerate(-scores_kfold):
    print(f"  {i+1:2d}   {s:.5f}")

print()
print("The walk-forward trajectory is the decision signal.")
print("Rising scores tell you: add a trend term, re-fit quarterly, or load the rate.")

## 5. Policy-year splits

When rate changes align to 1 January boundaries, `policy_year_split` gives clean year-aligned folds. No overlap between rate years in train and test.

In [ ]:
from insurance_cv import policy_year_split

py_splits = policy_year_split(
    df,
    date_col="inception_date",
    n_years_train=1,
    n_years_test=1,
    step_years=1,
)

print(f"Policy-year splits: {len(py_splits)} folds")
py_summary = split_summary(py_splits, df, date_col="inception_date")
print(py_summary)

## What you should see

- Walk-forward Poisson deviance is higher (looks worse) than random k-fold. This is correct -- k-fold is optimistic because of temporal leakage.
- The walk-forward per-fold trajectory rises as the test window advances into the trending period. This is the signal that tells you the model needs a trend adjustment.
- The leakage check passes: no forward-looking splits were generated.
- Policy-year folds align to clean 1 Jan boundaries.

The gap between k-fold and walk-forward CV scores is your "false confidence" metric. On portfolios with meaningful trend or seasonality it is typically 10-20% -- large enough to matter when you are choosing between models.

## Next steps

- **`accident_year_split`** -- for liability and PI lines where development triangle structure matters
- **`ibnr_buffer_months=12`** -- for longer-tail lines; increases gap between train and test
- Pass `InsuranceCV` directly to `GridSearchCV` for hyperparameter tuning without leakage

**GitHub:** https://github.com/burning-cost/insurance-cv  
**PyPI:** https://pypi.org/project/insurance-cv/